<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/10-cnn-keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:10000], y_train_full[:10000] # a little smaller for class
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

In [ ]:
X_train = X_train / 255.
X_valid = X_valid / 255.
X_test = X_test / 255.

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
n_rows = 4
n_cols = 10
plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_train[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        plt.title(class_names[y_train[index]])
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

## ANN

In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
model.add(tf.keras.layers.InputLayer(input_shape=[28,28]))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(300, activation="relu"))
model.add(tf.keras.layers.Dense(100, activation="relu"))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

In [ ]:
model.summary()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

In [ ]:
n_epochs=20
history = model.fit(X_train, y_train, epochs=n_epochs,
                    validation_data=(X_valid, y_valid))

In [ ]:
pd.DataFrame(history.history).plot(
    figsize=(8, 5), xlim=[0,(n_epochs-1)], grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc=(1.01,0))  # extra code
#save_fig("keras_learning_curves_plot")  # extra code
plt.show()

In [ ]:
model.evaluate(X_test, y_test)

## CNN

In [ ]:
from tensorflow import keras
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, Input
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import BatchNormalization
from keras.layers import LeakyReLU

In [ ]:
batch_size = 64
epochs = 20
num_classes = 10

# Note the use of leaky ReLUs because they attempt to fix the problem of
# dying Rectified Linear Units (ReLUs). This can happen when a large gradient flows
# through a ReLU neuron: it can cause the weights to update in such a way that the
# neuron will never activate on any data point again. If this happens, then the gradient
# flowing through the unit will forever be zero from that point on. Leaky ReLUs attempt to
#solve this: the function will not be zero but will instead have a small negative slope.

cnn_model = Sequential()
cnn_model.add(Conv2D(32, kernel_size=(3, 3),activation='linear',input_shape=(28,28,1),padding='same'))
cnn_model.add(LeakyReLU(alpha=0.1))
cnn_model.add(MaxPooling2D((2, 2),padding='same'))
cnn_model.add(Conv2D(64, (3, 3), activation='linear',padding='same'))
cnn_model.add(LeakyReLU(alpha=0.1))
cnn_model.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
cnn_model.add(Conv2D(128, (3, 3), activation='linear',padding='same'))
cnn_model.add(LeakyReLU(alpha=0.1))
cnn_model.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='linear'))
cnn_model.add(LeakyReLU(alpha=0.1))
cnn_model.add(Dense(num_classes, activation='softmax'))

In [ ]:
cnn_model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

print(cnn_model.summary())

## Train model

In [ ]:
n_epochs=20
history = cnn_model.fit(X_train, y_train, epochs=n_epochs,
                                  verbose=1, validation_data=(X_valid, y_valid))

## Evaluating on test data

In [ ]:
test_eval = cnn_model.evaluate(X_test, y_test, verbose=0)

print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

## Ploting accuracy and loss between training and validation data

In [ ]:
pd.DataFrame(history.history).plot(
    figsize=(8, 5), xlim=[0,(n_epochs-1)], grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc=(1.01,0))  # extra code
#save_fig("keras_learning_curves_plot")  # extra code
plt.show()

## Adding dropout to combat overfitting

In [ ]:
batch_size = 64
epochs = 20
num_classes = 10

cnn_model_2 = Sequential()
cnn_model_2.add(Conv2D(32, kernel_size=(3, 3),activation='linear',padding='same',input_shape=(28,28,1)))
cnn_model_2.add(LeakyReLU(alpha=0.1))
cnn_model_2.add(MaxPooling2D((2, 2),padding='same'))
cnn_model_2.add(Dropout(0.25))
cnn_model_2.add(Conv2D(64, (3, 3), activation='linear',padding='same'))
cnn_model_2.add(LeakyReLU(alpha=0.1))
cnn_model_2.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
cnn_model_2.add(Dropout(0.25))
cnn_model_2.add(Conv2D(128, (3, 3), activation='linear',padding='same'))
cnn_model_2.add(LeakyReLU(alpha=0.1))
cnn_model_2.add(MaxPooling2D(pool_size=(2, 2),padding='same'))
cnn_model_2.add(Dropout(0.4))
cnn_model_2.add(Flatten())
cnn_model_2.add(Dense(128, activation='linear'))
cnn_model_2.add(LeakyReLU(alpha=0.1))
cnn_model_2.add(Dropout(0.3))
cnn_model_2.add(Dense(num_classes, activation='softmax'))

cnn_model_2.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])
print(cnn_model_2.summary())

## Training with dropout

In [ ]:
history_2 = cnn_model_2.fit(X_train, y_train,
                                    epochs=n_epochs, verbose=1, validation_data=(X_valid, y_valid))

## Evaluating on test data

In [ ]:
# save first for potential re-use
#cnn_model_2.save("fashion_model_dropout.h5py")

test_eval1 = cnn_model_2.evaluate(X_test, y_test, verbose=1)

print('Test loss:', test_eval[0])
print('Test accuracy:', test_eval[1])

## Ploting accuracy and loss between training and validation data

In [ ]:
pd.DataFrame(history_2.history).plot(
    figsize=(8, 5), xlim=[0,(n_epochs-1)], grid=True, xlabel="Epoch",
    style=["r--", "r--.", "b-", "b-*"])
plt.legend(loc=(1.01,0))  # extra code
#save_fig("keras_learning_curves_plot")  # extra code
plt.show()

## Predicting labels

In [ ]:
predicted_classes = cnn_model_2.predict(X_test)

# Since predictions are floating point values, it will not be feasible to compare
#predicted labels with true test labels. First, round off output which will convert
#the float values into an integer. Then, use np.argmax() to select the index number
#which has a highest value in a row. For example, if prediction for test
#image is 0 1 0 0 0 0 0 0 0 0, output should be class label 1.

predicted_classes = np.argmax(np.round(predicted_classes), axis=1)
print(predicted_classes.shape, y_test.shape)

## Looking at a few correct predictions

In [ ]:
correct = np.where(predicted_classes == y_test)[0]
print("Found %d correct labels" % len(correct))
X_correct = X_test[correct]
y_correct = y_test[correct]
y_correct_pred = predicted_classes[correct]

In [ ]:
n_rows = 3
n_cols = 3
plt.figure(figsize=(n_cols * 1.5, n_rows * 1.5))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_correct[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        title = class_names[y_correct[index]] + '--' + class_names[y_correct_pred[index]]
        plt.title(title, fontsize=8)
plt.subplots_adjust(wspace=0.5, hspace=0.5)
plt.show()

## Looking at a few incorrect predictions

In [ ]:
incorrect = np.where(predicted_classes != y_test)[0]
print("Found %d incorrect labels" % len(incorrect))
X_incorrect = X_test[incorrect]
y_incorrect = y_test[incorrect]
y_incorrect_pred = predicted_classes[incorrect]

In [ ]:
n_rows = 3
n_cols = 3
plt.figure(figsize=(n_cols * 1.5, n_rows * 1.5))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_incorrect[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        title = class_names[y_incorrect[index]] + '--' + class_names[y_incorrect_pred[index]]
        plt.title(title, fontsize=8)
plt.subplots_adjust(wspace=0.5, hspace=0.5)
plt.show()

## Classification summary

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predicted_classes, target_names=class_names))